# EuroSAT CNN - Satellite Image Land-Use Classification

## Project Overview

This notebook trains and evaluates a custom PyTorch CNN for image-level land-use / land-cover classification on the EuroSAT RGB dataset. Each 64x64 RGB satellite image receives one of 10 class labels. It is not semantic segmentation.

The architecture, data split, optimizer, learning rate, and epoch count follow the supplied experiment. That experiment achieved 88.19% test accuracy; results can vary across hardware and library versions when rerun.


## Imports


In [ ]:
from pathlib import Path
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


## Configuration


In [ ]:
# Set this to the extracted EuroSAT_RGB directory containing the 10 class folders.
data_dir = Path("/path/to/EuroSAT_RGB")

batch_size = 32
num_epochs = 10
learning_rate = 0.001
random_seed = 42
num_workers = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

project_root = Path.cwd().resolve()
if not (project_root / "models").exists() and (project_root.parent / "models").exists():
    project_root = project_root.parent
results_dir = project_root / "results"
models_dir = project_root / "models"
results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)
model_path = models_dir / "eurosat_cnn_model.pth"

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

print(f"Using device: {device}")
print(f"Dataset directory: {data_dir}")


## Dataset Loading


In [ ]:
if not data_dir.is_dir():
    raise FileNotFoundError(
        "Set data_dir to the extracted EuroSAT_RGB directory before running this notebook."
    )

expected_classes = {
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
}
available_classes = {path.name for path in data_dir.iterdir() if path.is_dir()}
if available_classes != expected_classes:
    raise ValueError(
        "data_dir must contain the 10 EuroSAT RGB class folders. "
        f"Found: {sorted(available_classes)}"
    )


## Data Preprocessing


In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
classes = full_dataset.classes
num_classes = len(classes)

# Original experiment: deterministic 80/20 random train/test split.
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(random_seed),
)

pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_data, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_data, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory,
)

print("Classes:", classes)
print("Training images:", len(train_data))
print("Testing images:", len(test_data))


## Model Architecture


In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNNModel(num_classes=num_classes).to(device)
print(model)


## Training


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
training_losses = []
best_loss = float("inf")
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = 100.0 * running_correct / running_total
    training_losses.append(epoch_loss)

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_model_weights = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch + 1:02d}/{num_epochs} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%")

model.load_state_dict(best_model_weights)


## Training Results


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), training_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("CNN Training Loss")
plt.grid(True)
plt.tight_layout()
plt.savefig(results_dir / "training_loss.png", dpi=150)
plt.show()


Recorded original experiment: over 10 epochs, training accuracy increased from 59.75% to 89.27% and training loss fell from 1.1819 to 0.3286. The committed results image was extracted from the supplied notebook output. Rerunning this cell overwrites it with the current run's plot.


## Test Evaluation


In [ ]:
model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predictions = outputs.argmax(1)
        all_true.extend(labels.cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())

test_acc = accuracy_score(all_true, all_pred)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(classification_report(all_true, all_pred, target_names=classes, zero_division=0))


Recorded original experiment: test accuracy was 88.19% on 5,400 held-out images. See the report and README for the original per-class metrics.


## Confusion Matrix


In [ ]:
cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("EuroSAT Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(results_dir / "confusion_matrix.png", dpi=150)
plt.show()


The committed confusion matrix is the original notebook visualization. Rerun the evaluation cells before regenerating it for a new experiment.


## Sample Predictions


In [ ]:
def predict_image(image_path, model, transform, classes, device):
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(input_tensor), dim=1)
        confidence, index = torch.max(probabilities, dim=1)

    predicted_class = classes[index.item()]
    print(f"Predicted: {predicted_class} | Confidence: {confidence.item() * 100:.2f}%")
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"{predicted_class} ({confidence.item() * 100:.1f}%)")
    plt.axis("off")
    plt.show()


# Choose a local image, or uncomment these two lines for a dataset sample.
# sample_image = next((data_dir / "Forest").glob("*"))
# predict_image(sample_image, model, transform, classes, device)


The original run correctly predicted four displayed dataset samples: Forest (98.86%), Highway (99.79%), River (63.88%), and SeaLake (99.94%). The dataset images are intentionally excluded from this repository.


## Saving/Loading the Model


In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": classes,
        "class_to_idx": full_dataset.class_to_idx,
    },
    model_path,
)
print(f"Model saved to: {model_path}")

# Load the saved checkpoint in a fresh session.
loaded_checkpoint = torch.load(model_path, map_location=device, weights_only=True)
loaded_model = CNNModel(num_classes=len(loaded_checkpoint["class_names"])).to(device)
loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.eval()
print("Model loaded successfully.")
